# 40 · Explainability
Explains the zero-shot LLM decisions for the explanation sample of docs/12 §3.8: SHAP and LIME word attributions on the Yes/No log-odds, ERASER faithfulness (with "..." masking and with deletion), counterfactual rewrites (gender swap; slurs and profanity removed) on every scored post, and coding of the models' own self-explanations.
- **Inputs:** id lists written by the `99` post-hoc section (`OUTPUT_ROOT/20261023_all_all_llms_posthoc_s42/xai_ids/{dataset}-{task}-{model}-{split}.txt`). They hold ids only; which outcome cell (TP/FP/FN/TN) a post belongs to stays in `99`, so no test label reaches this notebook. Post texts come from `data/processed` (text only). Self-explanation runs (text backend, prompt variant X) come from a separate experiment run with notebook 20.
- **Outputs:** experiment folder `OUTPUT_ROOT/experiments/{YYYYMMDD-HHMMSS}_experiment_xai/` with one run per pair (`explanations.jsonl`, `counterfactual.jsonl`) and one coding run per self-explanation run (`codes.jsonl`, `human_coding_sheet.csv`), plus `manifest.csv` and `heartbeat.json`. The analysis by outcome is in `99`.
- **Sanity first:** `CONFIG["sample"] = N` explains N posts per pair with small budgets into `xai-sample`; check **SANITY OK**, then set `sample` to `None`.
- Self-explanations are model claims, not faithful explanations (Turpin et al. 2023); the paper contrasts them with the attribution and counterfactual evidence.
- No raw post text is printed.

In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"  # on Drive, so runs survive disconnects
HF_HOME = Path("/content/hf_cache") if IN_COLAB else PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)
sys.path.insert(0, str(PROJECT_ROOT))
if IN_COLAB:  # local copy of the processed data for speed; outputs still go to Drive
    shutil.copytree(DATA_ROOT / "processed", Path("/content/data/processed"), dirs_exist_ok=True)
    DATA_ROOT = Path("/content/data")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT  ", DATA_ROOT, "\nOUTPUT_ROOT", OUTPUT_ROOT, "\nHF_HOME    ", HF_HOME)

In [ ]:
# Same pins as 00 and requirements.txt; torch is Colab's build, logged not pinned
PINS = {"pandas": "3.0.6", "numpy": "2.5.4", "scikit-learn": "1.9.1", "pyarrow": "26.0.0", "pyyaml": "6.0.3",
        "transformers": "5.20.1", "pydantic": "2.13.5", "shap": "0.54.1", "lime": "0.2.0.1"}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS}
    print("If pip changed anything, restart the runtime, then run all again")
from importlib.metadata import version

off_pin = {p: version(p) for p in PINS if version(p) != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"

In [ ]:
# The only cell to edit between batches; rules are fixed in docs/12 §3.8
CONFIG = {
    "experiment": "xai",
    "purpose": "Explanations for the docs/12 §3.8 sample: SHAP, LIME, faithfulness, counterfactuals, coded self-explanations",
    "protocol_ref": "docs/12-protocol-addendum.md §3.8",
    "sample": 2,  # sanity: N posts per pair, SHAP max_evals 20, LIME 50 samples, into 'xai-sample'; None = full
    "ids_dir": "20261023_all_all_llms_posthoc_s42/xai_ids",  # under OUTPUT_ROOT; written by 99 (ids only)
    "shap_max_evals": 300,
    "lime_samples": 500,
    "lime_seeds": [42, 13, 87],  # all posts use 42; the first `lime_stability_posts` also use 13 and 87
    "lime_stability_posts": 20,
    "ks": [1, 3, 5],
    "n_random": 5,
    "seed": 42,
    "counterfactual": True,  # gender swap and slur removal on every scored post of each pair's split
    "selfexpl_experiment": None,  # folder name of the notebook-20 batch with backend text + prompt variant X
    "coder_model": "qwen3.5-4b",
    "human_fraction": 0.2,  # seeded share of rationales exported for the human check (kappa in 99)
    "codes": {
        "targets_women_or_gender": "says the text targets or demeans women or gender",
        "not_targeting_women": "says the text is not about women or not aimed at them",
        "offensive_language": "points to insults, slurs or profanity",
        "stereotype_or_inequality": "points to a stereotype, gender role or inequality claim",
        "reported_or_critical": "says the text reports, quotes or criticises sexism",
        "humour_or_sarcasm": "says the text is a joke, ironic or sarcastic",
        "needs_context": "says the meaning depends on missing context",
        "opinion_not_hate": "says it is an opinion, question or fact rather than sexism",
        "uninformative": "gives no usable reason",
        "other": "any other reason",
    },
    "heartbeat_s": 60,
}
SAMPLE = CONFIG["sample"] is not None

In [ ]:
# Token from Colab secrets (or the local env), never printed or pasted here
if IN_COLAB:
    from google.colab import userdata

    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print("HF_TOKEN set:", bool(HF_TOKEN))

In [ ]:
import gc
import json
import time
from datetime import datetime

import numpy as np
import pandas as pd

from src.prompts import ChatOptions, PromptTemplate
from src.schemas import get_schema
from src.utils import (
    LabelScorer, attribution_agreement, counterfactual_texts, experiment_date, faithfulness, lexicon_class,
    lime_explain, lime_stability, load_experiment_runs, load_lexicons, load_llm_registry, load_split, make_run_name,
    open_experiment, prepare_run_dir, resolve_revision, runs_dir, set_seeds, shap_explain, write_heartbeat,
)

REGISTRY = load_llm_registry(PROJECT_ROOT / "configs" / "llms.yaml")
LEX = load_lexicons()
EXP_DIR = open_experiment(
    OUTPUT_ROOT, CONFIG["experiment"], {k: v for k, v in CONFIG.items() if k not in ("purpose", "protocol_ref")},
    purpose=CONFIG["purpose"], protocol_ref=CONFIG["protocol_ref"], project_root=PROJECT_ROOT, sample=SAMPLE,
    ignore=("heartbeat_s",) + (("sample",) if SAMPLE else ()),
)
RUNS = runs_dir(EXP_DIR)
# One pair per id list: {dataset}-{task}-{model}-{split}.txt (task keys have no '-'; model names may)
PAIRS = []
for f in sorted((OUTPUT_ROOT / CONFIG["ids_dir"]).glob("*.txt")):
    if f.stem.endswith("-fnfp"):
        continue
    if not f.read_text().strip():
        print(f"skipped {f.name}: no ids (an outcome cell was empty)")
        continue
    parts = f.stem.split("-")
    PAIRS.append({"dataset": parts[0], "task": parts[1], "model": "-".join(parts[2:-1]), "split": parts[-1], "ids_file": f})
suffix = f"-sample-n{CONFIG['sample']}" if SAMPLE else ""
for p in PAIRS:
    p["run_name"] = make_run_name(experiment_date(EXP_DIR), p["dataset"], p["task"], p["model"],
                                  f"xai-{p['split']}{suffix}", CONFIG["seed"])
print(EXP_DIR, f"\n{len(PAIRS)} pairs:", [(p["dataset"], p["task"], p["model"], p["split"]) for p in PAIRS])

In [ ]:
from src.model import load_hf_model

budget = {"max_evals": 20, "lime": 50} if SAMPLE else {"max_evals": CONFIG["shap_max_evals"], "lime": CONFIG["lime_samples"]}
t0 = time.time()
for k, p in enumerate(PAIRS, 1):
    run_dir = RUNS / p["run_name"]
    if (run_dir / "complete.json").exists():
        continue
    write_heartbeat(EXP_DIR, {"state": "running", "n_pairs": len(PAIRS), "current": p["run_name"], "done": k - 1,
                              "elapsed_min": round((time.time() - t0) / 60, 1)})
    schema, spec = get_schema(p["dataset"], p["task"]), REGISTRY[p["model"]]
    ids = [i.strip() for i in p["ids_file"].read_text().splitlines() if i.strip()]
    ids = ids[: CONFIG["sample"]] if SAMPLE else ids
    # Text only; on test the loader reads no label column
    frame, data_file, _ = load_split(DATA_ROOT, p["dataset"], p["split"], schema)
    text = frame.set_index("id")["text_norm"]
    revision = resolve_revision(spec.repo_id, spec.revision, HF_TOKEN)
    config = {"run_name": p["run_name"], "kind": "xai", "dataset": p["dataset"], "task": p["task"], "split": p["split"],
              "model": p["model"], "model_name": p["model"], "model_id": spec.repo_id, "revision": revision,
              "seed": CONFIG["seed"], "ids_file": p["ids_file"].name, "n_examples": len(ids), "sample": CONFIG["sample"],
              "experiment": EXP_DIR.name, "output_file": "explanations.jsonl", **{k: CONFIG[k] for k in
              ("shap_max_evals", "lime_samples", "lime_seeds", "lime_stability_posts", "ks", "n_random")}}
    prepare_run_dir(run_dir, config)
    set_seeds(CONFIG["seed"])
    model, tok = load_hf_model(spec.repo_id, revision, token=HF_TOKEN, fp8_dequantize=spec.fp8_dequantize,
                               trust_remote_code=spec.trust_remote_code)
    scorer = LabelScorer(model, tok, PromptTemplate.load(schema), schema.labels()[0],
                         chat=ChatOptions(prefill=spec.prefill, chat_kwargs=spec.chat_kwargs))
    exp_path = run_dir / "explanations.jsonl"
    done = {json.loads(line)["id"] for line in exp_path.read_text().splitlines()} if exp_path.exists() else set()
    with exp_path.open("a") as f:
        for j, i in enumerate(ids):
            if i in done:
                continue
            sh = shap_explain(scorer, text[i], max_evals=budget["max_evals"])
            sh["lexicon_class"] = [lexicon_class(t, LEX) for t in sh["tokens"]]
            seeds = CONFIG["lime_seeds"] if j < CONFIG["lime_stability_posts"] else CONFIG["lime_seeds"][:1]
            limes = [lime_explain(scorer, text[i], num_samples=budget["lime"], seed=s) for s in seeds]
            rec = {"id": i, "shap": sh, "lime": limes, "faithfulness": faithfulness(scorer, sh, CONFIG["ks"],
                   CONFIG["n_random"], CONFIG["seed"]), **attribution_agreement(sh, limes[0]),
                   "lime_seed_rho": lime_stability(limes) if len(limes) > 1 else None}
            f.write(json.dumps(rec) + "\n")
            f.flush()
    if CONFIG["counterfactual"]:
        cf_path = run_dir / "counterfactual.jsonl"
        cf_done = {json.loads(line)["id"] for line in cf_path.read_text().splitlines()} if cf_path.exists() else set()
        posts = list(text.index[: CONFIG["sample"]]) if SAMPLE else list(text.index)
        with cf_path.open("a") as f:
            for i in posts:
                if i in cf_done:
                    continue
                rew = counterfactual_texts(text[i], LEX)
                todo = [text[i]] + [rew[c] for c in ("swap", "noslur") if rew[c] != text[i]]
                s = dict(zip(["orig"] + [c for c in ("swap", "noslur") if rew[c] != text[i]], scorer(todo)))
                f.write(json.dumps({"id": i, "s_orig": s["orig"], "s_swap": s.get("swap", s["orig"]),
                                    "s_noslur": s.get("noslur", s["orig"]), "s_swap_changed": "swap" in s,
                                    "s_noslur_changed": "noslur" in s}) + "\n")
    (run_dir / "complete.json").write_text(json.dumps({"n_predicted": len(ids), "n_forward_passes": scorer.n_calls,
                                                       "n_unscorable": scorer.n_nan,
                                                       "finished": datetime.now().isoformat(timespec="seconds")}, indent=2))
    (run_dir / "README.md").write_text(
        f"# {p['run_name']}\n\n**Purpose.** Explanations (docs/12 §3.8) for {p['model']} on {p['dataset']}/{p['task']} "
        f"{p['split']}: SHAP, LIME, ERASER faithfulness, counterfactual rewrites.\n**Ids.** `{p['ids_file'].name}` "
        f"({len(ids)} posts; outcome cells stay in 99).\n**Outcome.** `explanations.jsonl`, `counterfactual.jsonl`; "
        f"analysis by outcome in `99_final_eval`.\n")
    del model, scorer
    gc.collect()
    try:
        import torch

        torch.cuda.empty_cache()
    except Exception:
        pass
    print(f"[{k}/{len(PAIRS)}] {p['run_name']} done after {(time.time() - t0) / 60:.1f} min")
write_heartbeat(EXP_DIR, {"state": "finished", "n_pairs": len(PAIRS), "elapsed_min": round((time.time() - t0) / 60, 1)})

In [ ]:
# Code the self-explanations (text backend, variant X) into the fixed taxonomy; export a seeded 20% for the human check
from src.model.text import parse_yes_no
from src.utils import code_rationales

CODE_RUNS = []
if CONFIG["selfexpl_experiment"] is None:
    print("no self-explanation experiment set (CONFIG['selfexpl_experiment']): coding skipped")
else:
    src_exp = OUTPUT_ROOT / "experiments" / CONFIG["selfexpl_experiment"]
    selfexpl = load_experiment_runs([src_exp], "test", allow_sample=SAMPLE) + load_experiment_runs([src_exp], "dev", allow_sample=SAMPLE)
    spec = REGISTRY[CONFIG["coder_model"]]
    revision = resolve_revision(spec.repo_id, spec.revision, HF_TOKEN)
    coder, ctok = load_hf_model(spec.repo_id, revision, token=HF_TOKEN, fp8_dequantize=spec.fp8_dequantize,
                                trust_remote_code=spec.trust_remote_code)
    chat = ChatOptions(prefill=spec.prefill, chat_kwargs=spec.chat_kwargs)
    for r in selfexpl:
        name = make_run_name(experiment_date(EXP_DIR), r["dataset"], r["task"], r["model_name"],
                             f"codes-{r['split']}{suffix}", CONFIG["seed"])
        run_dir = RUNS / name
        CODE_RUNS.append(name)
        if (run_dir / "complete.json").exists():
            continue
        parsed = [(x["id"], *parse_yes_no(x["raw_output"] or "")) for x in r["records"]]
        config = {"run_name": name, "kind": "xai-codes", "dataset": r["dataset"], "task": r["task"], "split": r["split"],
                  "model": r["model_name"], "model_name": r["model_name"], "source_run": r["run_name"],
                  "coder_model": spec.repo_id, "coder_revision": revision, "codes": CONFIG["codes"], "seed": CONFIG["seed"],
                  "n_examples": len(parsed), "sample": CONFIG["sample"], "experiment": EXP_DIR.name, "output_file": "codes.jsonl"}
        prepare_run_dir(run_dir, config)
        coded = code_rationales(coder, ctok, [rat for _, _, rat in parsed], CONFIG["codes"], chat)
        with (run_dir / "codes.jsonl").open("w") as f:
            for (i, ans, rat), c in zip(parsed, coded):
                f.write(json.dumps({"id": i, "parsed_answer": ans, **c}) + "\n")
        rng = np.random.default_rng(CONFIG["seed"])
        pick = rng.choice(len(parsed), size=max(1, round(CONFIG["human_fraction"] * len(parsed))), replace=False)
        # The human codes blind to the model's code; saved as human_coding_filled.csv when done (kappa in 99)
        pd.DataFrame([{"id": parsed[j][0], "rationale": parsed[j][2], "human_code": ""} for j in sorted(pick)]).to_csv(
            run_dir / "human_coding_sheet.csv", index=False)
        (run_dir / "complete.json").write_text(json.dumps({"n_predicted": len(parsed),
                                                           "finished": datetime.now().isoformat(timespec="seconds")}))
        print(name, "coded", len(parsed))

In [ ]:
from src.utils import sanity_table, write_manifest

MANIFEST = write_manifest(EXP_DIR)  # rebuilt from the run folders; 99 reads runs only through it
print(MANIFEST.status.value_counts().to_dict())
if SAMPLE:
    SANITY = sanity_table(EXP_DIR, [p["run_name"] for p in PAIRS] + CODE_RUNS)
    print("SANITY OK" if SANITY.ok.all() else f"SANITY FAILED: {int((~SANITY.ok).sum())} runs, see the table")
    display(SANITY)

## Summary
TODO after the full run: pairs explained, forward passes, unscorable perturbations (`n_unscorable` in each `complete.json`), and coding coverage. All analysis by outcome cell is in `99_final_eval` (post-hoc section, `xai_*` tables).